# load data 

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/student_mental_health.csv")
df = df.drop(columns=["Student_ID"])

print("Shape after dropping ID:", df.shape)
df.head()

Shape after dropping ID: (2000, 16)


,Age,Gender,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Mental_Health_Risk
0,22,Male,3,6.6,17.6,2.91,6.9,7.5,Moderate,8.5,4.0,78.0,No,Poor,No,High
1,21,Male,4,5.6,22.4,2.92,7.6,11.5,Low,4.5,5.3,85.4,Yes,Average,No,Medium
2,22,Male,5,7.1,28.1,2.68,6.5,8.9,Low,4.8,6.1,79.5,No,Average,No,Medium
3,24,Male,1,4.5,14.8,2.23,5.5,7.6,High,3.7,8.0,73.9,No,Average,Yes,High
4,20,Female,1,6.7,17.2,2.69,4.3,7.0,Moderate,8.0,7.4,99.8,Yes,NaN,No,Medium


# check & handle anomalies

In [2]:
anomaly_mask = df["Social_Media_Hours"] > df["Screen_Time_Hours"]
print("Anomalies:", anomaly_mask.sum())

df["Screen_Time_Hours"] = df[["Screen_Time_Hours", "Social_Media_Hours"]].max(axis=1)

Anomalies: 37


# train-test split

In [3]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Mental_Health_Risk"])
y = df["Mental_Health_Risk"].map({"Low": 0, "Medium": 1, "High": 2})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (1600, 15) | Test: (400, 15)


# Fill missing values

In [4]:
num_cols = X_train.select_dtypes(include="number").columns
cat_cols = X_train.select_dtypes(exclude="number").columns

medians = X_train[num_cols].median()
modes = X_train[cat_cols].mode().iloc[0]

X_train[num_cols] = X_train[num_cols].fillna(medians)
X_test[num_cols] = X_test[num_cols].fillna(medians)

X_train[cat_cols] = X_train[cat_cols].fillna(modes)
X_test[cat_cols] = X_test[cat_cols].fillna(modes)

print("Remaining NaNs:", X_train.isna().sum().sum(), X_test.isna().sum().sum())

Remaining NaNs: 0 0


# encoding

In [5]:
from sklearn.preprocessing import StandardScaler

mappings = {
    "Physical_Activity": {"Low": 0, "Moderate": 1, "High": 2},
    "Diet_Quality": {"Poor": 0, "Average": 1, "Good": 2},
    "Part_Time_Job": {"No": 0, "Yes": 1},
    "Previous_Mental_Health_History": {"No": 0, "Yes": 1},
}
for col, mapping in mappings.items():
    X_train[col] = X_train[col].map(mapping)
    X_test[col] = X_test[col].map(mapping)

X_train = pd.get_dummies(X_train, columns=["Gender"], dtype=int)
X_test = pd.get_dummies(X_test, columns=["Gender"], dtype=int)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

X_train.head()

,Age,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Gender_Female,Gender_Male,Gender_Non-binary
521,0.416561,-1.224199,-0.042057,-1.462077,-0.852807,-0.092173,-0.095357,1,-0.259617,-0.021546,-0.520291,0,1,0,1,0,0
1393,-0.058152,-1.224199,1.227579,-1.375823,-1.696649,-0.715563,-0.399647,1,0.657234,-1.233064,-1.164891,1,0,1,1,0,0
1710,0.891274,1.058690,-0.495498,0.728769,-0.340475,0.085939,0.634939,0,0.225775,0.694351,1.556754,1,2,0,0,1,0
555,-0.532865,-0.463236,0.411385,-0.116518,-0.250063,0.353106,-0.217073,2,-0.637144,-0.627305,-0.907051,1,1,0,0,1,0
523,0.416561,1.058690,-0.314122,0.142243,-1.214454,-0.359340,-0.034499,1,1.304423,1.189972,-3.256259,0,2,1,1,0,0


# save

In [6]:
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)